# Elastic Net — Implementación Manual y Resultados
**Proyecto P1 Optimización · UTEC 2026**  
Dataset: *Housing Prices Lima, Peru* (Urbania)

Implementación de Elastic Net via **Coordinate Descent from scratch**, sin usar las rutinas internas de scikit-learn.

| Sección | Contenido |
|---|---|
| 1 | Librerías y carga del modelo |
| 2 | Carga de datos (preprocessing compartido) |
| 3 | Búsqueda de hiperparámetros (5-Fold CV) |
| 4 | Entrenamiento final |
| 5 | Validación contra scikit-learn |
| 6 | Coeficientes más importantes |
| 7 | Gráficas de resultados |
| 8 | Regularization Path |
| 9 | Predicción de propiedad nueva |

## 1. Librerías y carga del modelo

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# Importar módulos del proyecto
from src.preprocessing import get_housing_data
from src.elasticnet_manual import ElasticNetFromScratch

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 150,
})
BLUE, GREEN, RED, GRAY, LIGHT = "#2E75B6", "#70AD47", "#C00000", "#595959", "#F2F2F2"
print("Librerías cargadas correctamente.")

## 2. Carga de datos

Usamos `preprocessing.py` compartido para garantizar el **mismo split y estandarización** que el equipo de LASSO.

> ⚠️ Nadie hace el preprocesamiento por su cuenta. Esta única línea reemplaza todo el bloque de `read_excel`, `train_test_split` y `StandardScaler`.

In [ ]:
X_train, X_test, y_train, y_test, feature_cols, scaler = get_housing_data(
    "../data/housing_lima_final.xlsx"
)

## 3. Búsqueda de Hiperparámetros (5-Fold CV)

Elastic Net tiene dos hiperparámetros:
- **λ (lambda)**: intensidad de regularización. λ=0 → MCO sin penalización; λ grande → todos β→0
- **α (alpha)**: balance L1/L2. α=1 → LASSO puro; α=0 → Ridge puro

In [ ]:
lambdas = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
alphas  = [0.2, 0.5, 0.8]

best_mse, best_lambda, best_alpha = np.inf, None, None
cv_results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for alpha in alphas:
    for lam in lambdas:
        fold_mses = []
        for train_idx, val_idx in kf.split(X_train):
            Xtr, Xval = X_train[train_idx], X_train[val_idx]
            ytr, yval = y_train[train_idx], y_train[val_idx]
            model = ElasticNetFromScratch(lambda_=lam, alpha=alpha, max_iter=500, tol=1e-5)
            model.fit(Xtr, ytr)
            fold_mses.append(mean_squared_error(yval, model.predict(Xval)))
        mean_mse = np.mean(fold_mses)
        cv_results.append({"alpha": alpha, "lambda": lam, "cv_mse": mean_mse})
        if mean_mse < best_mse:
            best_mse, best_lambda, best_alpha = mean_mse, lam, alpha

print(f"Mejor lambda (λ): {best_lambda}")
print(f"Mejor alpha  (α): {best_alpha}")
print(f"Mejor CV-MSE:     {best_mse:.6f}")
print("\nTop 6 combinaciones:")
pd.DataFrame(cv_results).sort_values("cv_mse").head(6)

## 4. Entrenamiento Final

In [ ]:
model_scratch = ElasticNetFromScratch(
    lambda_=best_lambda, alpha=best_alpha, max_iter=2000, tol=1e-7
)
model_scratch.fit(X_train, y_train)

y_pred_train = model_scratch.predict(X_train)
y_pred_test  = model_scratch.predict(X_test)

mse_train  = mean_squared_error(y_train, y_pred_train)
mse_test   = mean_squared_error(y_test,  y_pred_test)
mae_test   = mean_absolute_error(y_test, y_pred_test)
r2_train   = r2_score(y_train, y_pred_train)
r2_test    = r2_score(y_test,  y_pred_test)
rmse_test  = np.sqrt(mse_test)
n_zeros    = np.sum(model_scratch.coef_ == 0)
n_nonzero  = np.sum(model_scratch.coef_ != 0)

print(f"  Iteraciones hasta convergencia: {model_scratch.n_iter_}")
print(f"  Coeficientes en cero:           {n_zeros} / {len(feature_cols)}")
print(f"  Coeficientes distintos de cero: {n_nonzero} / {len(feature_cols)}")
print(f"\n  {'Métrica':<20} {'Train':>10} {'Test':>10}")
print(f"  {'─'*20} {'─'*10} {'─'*10}")
print(f"  {'R²':<20} {r2_train:>10.4f} {r2_test:>10.4f}")
print(f"  {'MSE':<20} {mse_train:>10.6f} {mse_test:>10.6f}")
print(f"  {'RMSE':<20} {np.sqrt(mse_train):>10.6f} {rmse_test:>10.6f}")
print(f"  {'MAE':<20} {'—':>10} {mae_test:>10.6f}")

## 5. Validación contra scikit-learn

In [ ]:
model_sklearn = ElasticNet(
    alpha=best_lambda, l1_ratio=best_alpha,
    max_iter=10000, tol=1e-8, fit_intercept=True
)
model_sklearn.fit(X_train, y_train)

y_pred_sk     = model_sklearn.predict(X_test)
r2_sk         = r2_score(y_test, y_pred_sk)
mse_sk        = mean_squared_error(y_test, y_pred_sk)
coef_scratch  = model_scratch.coef_
coef_sk       = model_sklearn.coef_
max_coef_diff = np.max(np.abs(coef_scratch - coef_sk))

print(f"  {'Métrica':<20} {'From Scratch':>14} {'sklearn':>14} {'Diferencia':>12}")
print(f"  {'─'*20} {'─'*14} {'─'*14} {'─'*12}")
print(f"  {'R² Test':<20} {r2_test:>14.6f} {r2_sk:>14.6f} {abs(r2_test-r2_sk):>12.2e}")
print(f"  {'MSE Test':<20} {mse_test:>14.6f} {mse_sk:>14.6f} {abs(mse_test-mse_sk):>12.2e}")
print(f"  {'Intercepto':<20} {model_scratch.intercept_:>14.6f} {model_sklearn.intercept_:>14.6f} {abs(model_scratch.intercept_-model_sklearn.intercept_):>12.2e}")
print(f"  {'Max diff coef.':<20} {'':>14} {'':>14} {max_coef_diff:>12.2e}")
print(f"\n  ✓ Implementación validada correctamente" if max_coef_diff < 0.05
      else f"\n  ⚠ Diferencia notable: revisar tolerancia/iteraciones")

## 6. Coeficientes más importantes

In [ ]:
coef_df = pd.DataFrame({
    "Variable":     feature_cols,
    "Coef_Scratch": model_scratch.coef_,
    "Coef_sklearn": model_sklearn.coef_,
})
coef_df["Abs_Coef"] = np.abs(coef_df["Coef_Scratch"])
coef_df = coef_df.sort_values("Abs_Coef", ascending=False)

top15 = coef_df[coef_df["Coef_Scratch"] != 0].head(15)
elim  = coef_df[coef_df["Coef_Scratch"] == 0]

print(f"  {'Variable':<28} {'Coef. Scratch':>14} {'Coef. sklearn':>14}")
print(f"  {'─'*28} {'─'*14} {'─'*14}")
for _, row in top15.iterrows():
    print(f"  {row['Variable']:<28} {row['Coef_Scratch']:>14.6f} {row['Coef_sklearn']:>14.6f}")
print(f"\n  Variables eliminadas (β=0): {len(elim)}")
if len(elim) > 0:
    print("  " + ", ".join(elim["Variable"].tolist()[:8]) +
          (" ..." if len(elim) > 8 else ""))

## 7. Gráficas de resultados

In [ ]:
fig1, axes = plt.subplots(2, 2, figsize=(13, 10))
fig1.suptitle(
    f"Elastic Net — Housing Prices Lima, Peru\n"
    f"λ = {best_lambda}, α = {best_alpha}  |  "
    f"R² Test = {r2_test:.4f}  |  RMSE = {rmse_test:.4f}",
    fontsize=13, fontweight="bold", y=1.01
)

# ── Predicho vs Real ─────────────────────────────────────────────────────
ax = axes[0, 0]
ax.scatter(y_test, y_pred_test, alpha=0.35, s=12, color=BLUE, label="Observaciones")
lims = [min(y_test.min(), y_pred_test.min()) - 0.2,
        max(y_test.max(), y_pred_test.max()) + 0.2]
ax.plot(lims, lims, "--", color=RED, linewidth=1.5, label="Predicción perfecta")
ax.set_xlabel("log(Precio) real"); ax.set_ylabel("log(Precio) predicho")
ax.set_title("Predicho vs Real"); ax.legend(fontsize=8); ax.set_aspect("equal")

# ── Residuos ─────────────────────────────────────────────────────────────
ax = axes[0, 1]
residuos = y_test - y_pred_test
ax.scatter(y_pred_test, residuos, alpha=0.35, s=12, color=GREEN)
ax.axhline(0, color=RED, linewidth=1.5, linestyle="--")
ax.set_xlabel("log(Precio) predicho"); ax.set_ylabel("Residuo")
ax.set_title("Gráfico de Residuos")
ax.text(0.97, 0.05, f"RMSE = {rmse_test:.4f}\nMAE  = {mae_test:.4f}",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor=LIGHT))

# ── Convergencia ─────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(model_scratch.loss_history_, color=BLUE, linewidth=1.8)
ax.set_xlabel("Iteración"); ax.set_ylabel("Función de pérdida")
ax.set_title(f"Convergencia del Coordinate Descent\n({model_scratch.n_iter_} iteraciones)")
ax.set_yscale("log")

# ── Top 12 coeficientes ───────────────────────────────────────────────────
ax = axes[1, 1]
top_plot   = top15.head(12)
colors_bar = [GREEN if c > 0 else RED for c in top_plot["Coef_Scratch"]]
ax.barh(range(len(top_plot)), top_plot["Coef_Scratch"],
        color=colors_bar, edgecolor="white", height=0.7)
ax.set_yticks(range(len(top_plot)))
ax.set_yticklabels(top_plot["Variable"], fontsize=8)
ax.axvline(0, color=GRAY, linewidth=0.8)
ax.set_xlabel("Coeficiente")
ax.set_title("Top 12 Coeficientes\n(variables seleccionadas por Elastic Net)")
ax.invert_yaxis()

plt.tight_layout()
plt.savefig("/content/fig1_resultados.png", bbox_inches="tight", dpi=150)
plt.show()
print("✓ fig1_resultados.png guardada")

## 8. Regularization Path

Muestra cómo los coeficientes cambian (y son llevados a cero) a medida que λ aumenta.
Es la visualización clave para entender la selección de variables de Elastic Net.

In [ ]:
lambdas_path = np.logspace(-3, 0, 40)
coef_path    = []
for lam in lambdas_path:
    m = ElasticNetFromScratch(lambda_=lam, alpha=best_alpha, max_iter=500, tol=1e-5)
    m.fit(X_train, y_train)
    coef_path.append(m.coef_.copy())
coef_path = np.array(coef_path)

opt_idx    = np.argmin(np.abs(lambdas_path - best_lambda))
top8_idx   = np.argsort(np.abs(coef_path[opt_idx]))[-8:]
top8_names = [feature_cols[i] for i in top8_idx]

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
fig2.suptitle(f"Elastic Net — Regularization Path  (α = {best_alpha})",
              fontsize=13, fontweight="bold")

ax = axes2[0]
cmap = plt.cm.tab10
for i, (idx, name) in enumerate(zip(top8_idx, top8_names)):
    ax.plot(lambdas_path, coef_path[:, idx],
            label=name, color=cmap(i / 8), linewidth=1.8)
ax.axvline(best_lambda, color=RED, linestyle="--", linewidth=1.5,
           label=f"λ óptimo = {best_lambda}")
ax.set_xscale("log"); ax.set_xlabel("λ (escala log)"); ax.set_ylabel("Valor del coeficiente")
ax.set_title("Camino de Regularización\n(top 8 variables)")
ax.legend(fontsize=7, loc="upper right")

ax = axes2[1]
n_active = [np.sum(coef_path[i] != 0) for i in range(len(lambdas_path))]
ax.plot(lambdas_path, n_active, color=BLUE, linewidth=2)
ax.axvline(best_lambda, color=RED, linestyle="--", linewidth=1.5,
           label=f"λ óptimo = {best_lambda}")
ax.set_xscale("log"); ax.set_xlabel("λ (escala log)")
ax.set_ylabel("Número de variables activas (β ≠ 0)")
ax.set_title("Selección de Variables vs λ"); ax.legend(fontsize=9)

# ── From Scratch vs sklearn ──────────────────────────────────────────────
fig3, axes3 = plt.subplots(1, 2, figsize=(13, 5))
fig3.suptitle("Elastic Net — Validación: From Scratch vs sklearn",
              fontsize=13, fontweight="bold")

ax = axes3[0]
ax.scatter(coef_sk, coef_scratch, alpha=0.6, s=20, color=BLUE)
lim = max(np.abs(coef_sk).max(), np.abs(coef_scratch).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], "--", color=RED, linewidth=1.5, label="Igualdad perfecta")
ax.set_xlabel("Coeficientes sklearn"); ax.set_ylabel("Coeficientes From Scratch")
ax.set_title("Comparación de Coeficientes\n(cada punto = una variable)")
ax.legend(fontsize=9)
ax.text(0.05, 0.95, f"Max diferencia: {max_coef_diff:.2e}",
        transform=ax.transAxes, fontsize=9, verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor=LIGHT))

ax = axes3[1]
metricas = ["R² Test", "MSE Test", "RMSE Test"]
vals_sc  = [r2_test,  mse_test,  rmse_test]
vals_sk  = [r2_sk,    mse_sk,    np.sqrt(mse_sk)]
x = np.arange(len(metricas)); width = 0.35
bars1 = ax.bar(x - width/2, vals_sc, width, label="From Scratch", color=BLUE, edgecolor="white")
bars2 = ax.bar(x + width/2, vals_sk, width, label="sklearn",       color=GREEN, edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(metricas)
ax.set_title("Comparación de Métricas"); ax.legend(fontsize=9)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
fig2.savefig("/content/fig2_regularization_path.png", bbox_inches="tight", dpi=150)
fig3.savefig("/content/fig3_validacion.png", bbox_inches="tight", dpi=150)
plt.show()
print("✓ fig2_regularization_path.png guardada")
print("✓ fig3_validacion.png guardada")

## 9. Predicción para una Propiedad Nueva

**Rangos de referencia del dataset:**

| Variable | Mín | Máx | Mediana | Notas |
|---|---|---|---|---|
| Area_constr_m2 | 1 | 3640 | 296 | m² |
| Dormitorios | 1 | 5 | 4 | 5 = "5 o más" |
| NroBanios | 1 | 5 | 4 | |
| Cocheras | 0 | 8 | 2 | 0 = sin cochera |
| Antiguedad | 1 | 9 | 3 | 1=nueva … 9=muy antigua |
| Estado_inmueble | 1 | 5 | 3 | 1=remodelar … 5=excelente |

**Coordenadas de referencia:**
- La Molina: lat=-12.08, lon=-76.93
- San Isidro: lat=-12.10, lon=-77.05
- Miraflores: lat=-12.12, lon=-77.03
- Santiago de Surco: lat=-12.14, lon=-76.99

In [ ]:
nueva_propiedad = {
    # Rango: 1–3640 m² | Mediana: 296 m²
    "Area_constr_m2"    : 200,
    # Rango: 42–9663 m² | si no sabes, pon igual a Area_constr_m2
    "Area_total_m2"     : 300,
    # Rango: 1–5  (5 = "5 o más")
    "Dormitorios"       : 4,
    # Rango: 1–5
    "NroBanios"         : 3,
    # Rango: 1–8
    "Nro_pisos"         : 2,
    # Rango: 0–8  (0 = sin cochera)
    "Cocheras"          : 2,
    # 1=nueva, 2=casi nueva, 3=usada buen estado, 4=media, 5-9=antigua
    "Antiguedad"        : 3,
    # 1=A remodelar, 2=Regular, 3=Bueno, 4=Muy bueno, 5=Excelente
    "Estado_inmueble"   : 4,
    # La Molina: lat=-12.08, lon=-76.93
    "latitud"           : -12.08,
    "longitud"          : -76.93,
    # Variables engineered (se calculan automáticamente abajo)
    "area_log"           : None,
    "ratio_banos_dorm"   : None,
    "ratio_cocheras_area": None,
    # Amenities: 1 = tiene, 0 = no tiene
    "Cuarto de servicio": 1, "Deposito": 1, "Terraza": 0,
    "Sala de estar": 1, "Patio": 0, "Comedor diario": 1,
    "Comedor": 1, "Banio de servicio": 1, "Jardín Interno": 1,
    "Walking Closet": 1, "Escritorio": 0, "Cocina": 1,
    "Lavandería": 1, "Balcon": 0, "Sala": 1, "Closet": 1,
    "Banio de visitas": 1, "Agua": 1, "Guardianía": 1,
    "Internet": 1, "Luz": 1, "Cable": 0,
    "Servicio de Limpieza": 0, "Conexion a gas": 1,
    "Sistema de seguridad": 1, "Telefono": 0, "Areadeportiva": 0,
    "Piscina": 0, "Jardín": 1, "Áreas verdes": 1,
    "Hall de ingreso": 1, "Areade BBQ": 0, "Gimnasio": 0,
    "Colegios cercanos": 1, "Cerca al mar": 0,
    "Centros comerciales cercanos": 1, "Parques cercanos": 1,
    "Acceso personas discapacidad": 0, "Desagaue": 1,
    "Jacuzzi": 0, "Chimenea": 0, "Intercomunicador": 0,
    "Cerco Electrico": 1, "Parrilla": 0, "Aire acondicionado": 0,
    "Amoblado": 0, "Equipado": 0, "Terma": 1,
    "Portero electrico": 0, "Cocina con reposteros": 1,
    # Distrito: pon 1 solo en el que corresponda, el resto en 0
    "dist_LaMolina": 1, "dist_SantiagoDeSurco": 0,
    "dist_Asia": 0, "dist_SanIsidro": 0, "dist_Miraflores": 0,
    "dist_Chorrillos": 0, "dist_SanBorja": 0,
}

# Calcular variables engineered automáticamente
nueva_propiedad["area_log"]            = np.log(nueva_propiedad["Area_constr_m2"])
nueva_propiedad["ratio_banos_dorm"]    = nueva_propiedad["NroBanios"] / nueva_propiedad["Dormitorios"]
nueva_propiedad["ratio_cocheras_area"] = nueva_propiedad["Cocheras"] / nueva_propiedad["Area_constr_m2"]

X_nueva        = np.array([[nueva_propiedad[col] for col in feature_cols]])
X_nueva_scaled = scaler.transform(X_nueva)

pred_log    = model_scratch.predict(X_nueva_scaled)[0]
pred_usd    = np.exp(pred_log)
precio_bajo = np.exp(pred_log - mae_test)
precio_alto = np.exp(pred_log + mae_test)

print("=" * 55)
print("  PREDICCIÓN — PROPIEDAD NUEVA")
print("=" * 55)
print(f"  Distrito        : La Molina")
print(f"  Área construida : {nueva_propiedad['Area_constr_m2']} m²")
print(f"  Dormitorios     : {nueva_propiedad['Dormitorios']}")
print(f"  Baños           : {nueva_propiedad['NroBanios']}")
print(f"  Cocheras        : {nueva_propiedad['Cocheras']}")
print(f"  Estado          : {nueva_propiedad['Estado_inmueble']}/5")
print()
print(f"  Precio estimado : ${pred_usd:>12,.0f} USD")
print(f"  Rango probable  : ${precio_bajo:>12,.0f} — ${precio_alto:,.0f} USD")
print("=" * 55)